# Causal Trace: Alt

Runs the existing alternative trace mode with explicit candidate-layer visualization.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hydra
import matplotlib.pyplot as plt
import pandas as pd
from src.causal_trace.prototype import run_alt_trace

MODEL = 'gpt2-large'
NUM_PROMPTS = 1
NUM_NOISE = 10
RESTORE_POSITION = 'subject_last'


In [ ]:
with hydra.initialize_config_dir(config_dir=str(ROOT / 'src' / 'config'), version_base=None):
    cfg = hydra.compose(
        config_name='latium',
        overrides=[
            'command=alt_trace',
            f'model={MODEL}',
            f'generation.num_of_runs={NUM_PROMPTS}',
            f'tracing.num_noise_samples={NUM_NOISE}',
            f'tracing.restore_position={RESTORE_POSITION}',
        ],
    )

out_dir = Path(run_alt_trace(cfg))
scores = pd.read_csv(out_dir / 'wide_layer_scores.csv')
scores[['prompt_id', 'subject', 'target', 'trace_reliable', 'fallback_reason', 'candidate_layers']]


In [ ]:
layer_cols = [col for col in scores.columns if col.startswith('layer_')]
y = scores.loc[0, layer_cols].astype(float).to_numpy()
candidates = [int(x) for x in str(scores.loc[0, 'candidate_layers']).split() if x]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(y)), y, marker='o')
for layer in candidates:
    ax.axvline(layer, color='tab:red', linestyle='--', alpha=0.7)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean indirect effect')
ax.set_title('Alt trace candidates')
plt.show()
